# 22-26 · SQL и его отображение в ORM

Практика к разделу [«ORM: работа с базой через объекты Python»](../../site/chapters/glava-22/22-26-orm.html).

## Цель

Проверить, что запрос через игрушечный ORM-объект и прямой SQL-запрос дают одинаковый результат — на маленьком, полностью реальном примере sqlite3.

## Рабочий пример

In [1]:
import sqlite3

baza = sqlite3.connect(":memory:")
baza.row_factory = sqlite3.Row
baza.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
baza.executemany(
    "INSERT INTO tasks (title, done) VALUES (?, ?)",
    [("A", 0), ("B", 1), ("C", 0)],
)
baza.commit()


class ProstoyORM:
    """Игрушечная имитация одного метода ORM — под капотом всё равно SQL."""

    def __init__(self, soedinenie):
        self.soedinenie = soedinenie

    def nevypolnennye(self):
        return self.soedinenie.execute(
            "SELECT id, title, done FROM tasks WHERE done = 0 ORDER BY id"
        ).fetchall()


orm = ProstoyORM(baza)
cherez_orm = orm.nevypolnennye()
cherez_sql = baza.execute("SELECT id, title, done FROM tasks WHERE done = 0 ORDER BY id").fetchall()

print([dict(s) for s in cherez_orm])

[{'id': 1, 'title': 'A', 'done': 0}, {'id': 3, 'title': 'C', 'done': 0}]


## Проверка результата

In [2]:
assert [dict(s) for s in cherez_orm] == [dict(s) for s in cherez_sql]
assert [s["title"] for s in cherez_orm] == ["A", "C"]
print("Верно: ORM-обёртка и прямой SQL-запрос вернули одинаковый результат — под капотом ORM выполняет тот же SQL.")

Верно: ORM-обёртка и прямой SQL-запрос вернули одинаковый результат — под капотом ORM выполняет тот же SQL.
